<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 03 · 同时做两个项目，如何避免串台

同一位开发者维护两个导入器：国内订单使用人民币，海外订单使用美元。两边都有 `currency` 字段，
但它们的约定不能混在一起。我们会给两个项目提出同一个问题，看 Server 如何根据 Scope 选择数据。

在前两篇，Scope 只是初始化中的一个 ID。这一篇我们停下来，认真看看它怎样组织工作范围。

**这一篇的收获：** 创建独立 Scope，验证同名知识的分区，并区分组织层级、显式上下文引用和访问控制。

**运行准备：** 从 [教程入口](README.md) 安装依赖并启动 Jupyter。每篇都带有自己的数据，可以独立运行。
本篇不需要模型或 API Key。
建议先逐格运行，读完输出再继续；完整重跑时使用 **Restart Kernel & Run All**。

## 先准备一个自己的实验空间

下面的辅助代码只负责启动本地 Server、建立 Client 和整理输出。默认每次完整运行使用新的 SQLite 数据库；选择 OceanBase 时，使用专用测试库并为本次实验创建新的 Scope。
后端设置见 [README](README.md#使用-oceanbase-运行)。关键的写入、检索、审核与交接调用会直接写在后面的单元格里。


本篇通过 `remember_memory` 写入 Scope 的日常 Memory，使后续搜索和上下文准备读取同一份知识。独立制品的创建、版本管理与日常记忆的关系见 [接口选择](DESIGN.md#接口选择)。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))

if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("03", features=())
client = lab.client
assert client is not None

现在创建本篇的项目 Scope。`title` 是给人看的名称，真正用于调用的是 Server 返回的 `scope_id`。
你可以改变标题；不要自己根据标题或目录拼出一个 Scope ID。


In [ ]:
from powercontext.http import CreateScopeRequest

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 03",
        summary="第 03 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-03",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 为另一个项目创建 Scope

本篇开始时的 Scope 作为国内项目。再建一个海外项目。两个 Scope 都由 Server 分配身份；
同样的创建请求重复发送时，`idempotency_key` 可以帮助调用方恢复同一个创建结果。


In [ ]:
overseas_request = CreateScopeRequest(
    title="海外订单 CSV 导入器",
    summary="美元订单约定",
    idempotency_key=f"{lab.run_id}:overseas-orders",
)
overseas = await client.create_scope(overseas_request)
repeated = await client.create_scope(overseas_request)
assert repeated.scope_id == overseas.scope_id
assert overseas.scope_id != scope_id
table([{"项目": scope.title, "Scope": scope_id}, {"项目": overseas.title, "Scope": overseas.scope_id}])

## 2. 相同字段，各自保存自己的决定

除了 `scope_id`，两个请求采用相同写法。这样后续应用可以复用一份调用逻辑，把范围选择交给可信应用层。


In [ ]:
from powercontext.http import RememberMemoryRequest, SearchMemoryRequest

for selected, text in [
    (scope_id, "currency: 国内订单使用 CNY，金额单位为人民币分。"),
    (overseas.scope_id, "currency: 海外订单使用 USD，金额单位为美分。"),
]:
    await client.remember_memory(
        RememberMemoryRequest(
            scope_id=selected,
            kind="decision",
            text=text,
            reason="各项目已经确认的币种",
        )
    )

## 3. 给两个项目提出同一个问题

查询词都使用 `currency`，只切换 Scope。观察正文和引用，确认国内项目不会因为字段名相同而召回海外约定。


In [ ]:
domestic_result = await client.search_memory(SearchMemoryRequest(scope_id=scope_id, query="currency", mode="fts"))
overseas_result = await client.search_memory(
    SearchMemoryRequest(scope_id=overseas.scope_id, query="currency", mode="fts")
)
table([
    *[{"项目": "国内", "命中": hit.text} for hit in domestic_result.hits],
    *[{"项目": "海外", "命中": hit.text} for hit in overseas_result.hits],
])
assert domestic_result.hits and overseas_result.hits
assert all("USD" not in hit.text for hit in domestic_result.hits)
assert all("CNY" not in hit.text for hit in overseas_result.hits)

## 4. 子项目会自动继承父项目知识吗

我们在国内项目下创建一个迁移子项目。`parent_scope_id` 表达组织关系；它本身不会让父级 Memory
自动流入子项目的上下文。需要共享内容时，应明确设计 `context_references`，而不是依赖名称或层级猜测。
这个例子不设置任何上下文引用。


In [ ]:
child = await client.create_scope(
    CreateScopeRequest(
        title="国内订单迁移",
        summary="只表达组织层级，不配置上下文引用",
        parent_scope_id=scope_id,
        idempotency_key=f"{lab.run_id}:domestic-migration",
    )
)
child_result = await client.search_memory(SearchMemoryRequest(scope_id=child.scope_id, query="currency", mode="fts"))
assert child_result.hits == []
show({"父级": child.parent_scope_id, "子项目命中数": len(child_result.hits)})

## 轮到你：创建第三个项目

第三个项目使用 EUR。把它保存到一个新的 Scope，再确认原来两个项目的搜索结果不变。
下面的参考答案刻意复用同一个查询词。


In [ ]:
europe = await client.create_scope(
    CreateScopeRequest(
        title="欧洲订单导入器",
        summary="欧元订单约定",
        idempotency_key=f"{lab.run_id}:europe-orders",
    )
)
await client.remember_memory(
    RememberMemoryRequest(
        scope_id=europe.scope_id,
        kind="decision",
        text="currency: 欧洲订单使用 EUR。",
        reason="练习",
    )
)
result = await client.search_memory(SearchMemoryRequest(scope_id=europe.scope_id, query="currency", mode="fts"))
assert result.hits and all("EUR" in hit.text for hit in result.hits)
show({"第三个项目": [hit.text for hit in result.hits]})

## 用到真实应用时

本篇的本地 Server 只监听回环地址，并关闭访问控制，方便你独立实验。它证明的是按 Scope 选择数据的行为。
远程应用还需要身份验证和授权，Scope ID 本身不授予访问权。不要让模型随意挑选其他用户的 Scope。

项目路径、会话 ID 可以作为 binding 的输入，让应用恢复稳定的 Scope；它们本身也不是 Scope ID。
需要这条路线时，可以继续阅读 [核心概念](../../docs/zh/docs/get-started/core-concepts.md)。


## 带着结果离开

相同关键词可以在不同工作范围内表达不同知识。你已经验证了分区效果，也验证了单纯的父子层级不会自动扩大上下文可见性。

下面关闭本篇的 Client 和 Server。实验文件仍留在教程的 `.powercontext/` 子目录，便于检查；
清理方法见 [README](README.md#清理实验数据)。如果在中途停止，请运行这个单元格，或关闭 Kernel。

下一篇：[一次提问究竟需要多少上下文](04_prepared_context.ipynb)。


In [ ]:
await lab.close()
print("本篇 Server 已关闭。")